In [1]:
import pandas as pd
import numpy as np

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)

from sklearn.linear_model import LinearRegression

from sklearn.ensemble import RandomForestRegressor

from xgboost import XGBRegressor

from statsmodels.tsa.holtwinters import (
    SimpleExpSmoothing,
    Holt,
    ExponentialSmoothing
)

from statsmodels.tsa.arima.model import ARIMA

from statsmodels.tsa.statespace.sarimax import SARIMAX

import warnings
warnings.filterwarnings("ignore")

In [4]:
df = pd.read_csv(
    "final_data_for_vehicle_forecasting.csv"
)

df['Month'] = pd.to_datetime(
    df['Month'],
    dayfirst=True
)

print(df.shape)

print(df['Month'].min())
print(df['Month'].max())

(49680, 6)
2023-04-01 00:00:00
2025-03-01 00:00:00


In [46]:
def evaluate(actual, forecast):

    actual = np.array(actual)
    forecast = np.array(forecast)

    mae = mean_absolute_error(
        actual,
        forecast
    )

    rmse = np.sqrt(
        mean_squared_error(
            actual,
            forecast
        )
    )

    mask = actual != 0

    if mask.sum() == 0:

        return mae, rmse, np.nan, np.nan

    mape = (
        np.mean(
            np.abs(
                (
                    actual[mask]
                    -
                    forecast[mask]
                )
                /
                actual[mask]
            )
        )
        * 100
    )

    accuracy = max(
        0,
        100 - mape
    )

    return mae, rmse, mape, accuracy

In [47]:
def moving_average_forecast(train):

    return [train.tail(3).mean()] * 3

In [48]:
def ses_forecast(train):

    model = SimpleExpSmoothing(train)

    fit = model.fit()

    return fit.forecast(3)

In [49]:
def holt_forecast(train):

    model = Holt(train)

    fit = model.fit()

    return fit.forecast(3)

In [50]:
def hw_forecast(train):

    model = ExponentialSmoothing(
        train,
        trend='add'
    )

    fit = model.fit()

    return fit.forecast(3)

In [51]:
def arima_forecast(train):

    model = ARIMA(
        train,
        order=(1,1,1)
    )

    fit = model.fit()

    return fit.forecast(3)

In [52]:
def sarima_forecast(train):

    model = SARIMAX(
        train,
        order=(1,1,1),
        seasonal_order=(1,1,1,12)
    )

    fit = model.fit(
        disp=False
    )

    return fit.forecast(3)

In [53]:
forecast_models = {

    "Moving Average":
        moving_average_forecast,

    "SES":
        ses_forecast,

    "Holt":
        holt_forecast,

    "Holt Winters":
        hw_forecast,

    "ARIMA":
        arima_forecast,

    "SARIMA":
        sarima_forecast
}

In [54]:
results = []

forecast_details = []

In [55]:
for halb in df['Halb'].unique():

    df_halb = df[
        df['Halb'] == halb
    ]

    for engine in df_halb['engine_type'].unique():

        df_engine = df_halb[
            df_halb['engine_type']
            == engine
        ]

        for model in df_engine[
            'map_model'
        ].unique():

            df_model = df_engine[
                df_engine['map_model']
                == model
            ]

            for vehicle in df_model[
                'Vehicle_Type'
            ].unique():

                temp = df_model[
                    df_model['Vehicle_Type']
                    == vehicle
                ]

                series = (
                    temp.groupby('Month')
                    ['total_demand']
                    .sum()
                    .sort_index()
                )

                train = series.loc[
                    :'2024-12-01'
                ]

                test = series.loc[
                    '2025-01-01':
                    '2025-03-01'
                ]

                if len(train) < 12:
                    continue

                if len(test) != 3:
                    continue

                for model_name, model_func in (
                    forecast_models.items()
                ):

                    try:

                        forecast = model_func(
                            train
                        )

                        mae, rmse, mape, acc = (
                            evaluate(
                                test,
                                forecast
                            )
                        )

                        # Month-wise forecast details

                        for month, actual, pred in zip(
                            test.index,
                            test.values,
                            forecast
                        ):

                            forecast_details.append([

                                halb,
                                engine,
                                model,
                                vehicle,

                                model_name,

                                month,

                                actual,

                                pred,

                                actual - pred

                            ])

                        results.append([

                            halb,
                            engine,
                            model,
                            vehicle,

                            model_name,

                            mae,
                            rmse,
                            mape,
                            acc

                        ])

                    except:

                        continue

In [56]:
results_df = pd.DataFrame(

    results,

    columns=[

        'Halb',
        'Engine',
        'Model',
        'Vehicle_Type',

        'Forecast_Model',

        'MAE',
        'RMSE',
        'MAPE',
        'Accuracy'
    ]
)

In [57]:
forecast_details_df = pd.DataFrame(

    forecast_details,

    columns=[

        'Halb',
        'Engine',
        'Model',
        'Vehicle_Type',

        'Forecast_Model',

        'Month',

        'Actual_Demand',

        'Forecast_Demand',

        'Difference'

    ]

)

In [58]:
model_summary = (

    results_df
    .groupby(
        'Forecast_Model'
    )[
        ['MAE',
         'RMSE',
         'MAPE',
         'Accuracy']
    ]
    .mean()

    .sort_values(
        'Accuracy',
        ascending=False
    )
)

model_summary

,MAE,RMSE,MAPE,Accuracy
Forecast_Model,,,,
ARIMA,3.789408,4.544382,94.872078,26.374195
Holt Winters,3.979814,4.746577,90.733798,25.917659
SES,3.396911,4.182625,90.578277,24.458292
Moving Average,3.473698,4.250268,94.300064,22.846288
Holt,3.681303,4.462875,99.074444,22.120181
SARIMA,6.059157,7.242885,140.840742,17.232838


In [59]:
results_df.to_excel(
    "hierarchical_forecasting_results.xlsx",
    index=False
)

In [60]:
model_summary.to_excel(
    "model_summary.xlsx"
)

In [61]:
best_model_per_segment = (

    results_df
    .sort_values(
        'Accuracy',
        ascending=False
    )
    .groupby(
        [
            'Halb',
            'Engine',
            'Model',
            'Vehicle_Type'
        ]
    )
    .first()
    .reset_index()
)

best_model_per_segment.to_excel(
    "best_model_per_segment.xlsx",
    index=False
)

In [62]:
forecast_details_df.to_excel(
    writer,
    sheet_name="Forecast_Details",
    index=False
)

In [63]:
forecast_details_df.to_csv(
    "forecast_details.csv",
    index=False
)

print("CSV Saved Successfully")

CSV Saved Successfully


In [64]:
print(len(results))

12420


In [65]:
print(len(forecast_details))

37260


In [66]:
print(results_df.shape)

(12420, 9)


In [67]:
print(forecast_details_df.shape)

(37260, 9)


In [68]:
count = 0

for halb in df['Halb'].unique():

    df_halb = df[df['Halb'] == halb]

    for engine in df_halb['engine_type'].unique():

        df_engine = df_halb[
            df_halb['engine_type'] == engine
        ]

        for model in df_engine['map_model'].unique():

            df_model = df_engine[
                df_engine['map_model'] == model
            ]

            for vehicle in df_model['Vehicle_Type'].unique():

                temp = df_model[
                    df_model['Vehicle_Type']
                    == vehicle
                ]

                series = (
                    temp.groupby('Month')
                    ['total_demand']
                    .sum()
                    .sort_index()
                )

                train = series.loc[:'2024-12-01']
                test = series.loc['2025-01-01':'2025-03-01']

                if len(train) < 12:
                    continue

                if len(test) != 3:
                    continue

                count += 1

print(count)

2070
